# Inventory Validator API Frontend

This notebook is a lightweight notebook-side frontend for the FastAPI app.

It now mirrors the incremental processing flow as far as Jupyter can reasonably deliver:
- upload the CSV and create the async job;
- poll the job status with live notebook redraws;
- show partial preview data when the backend marks it as reliable;
- keep the PDF download restricted to the final consolidated result.

It assumes:
- the API is already running locally;
- you will point `CSV_PATH` to a real CSV file;
- `httpx`, `pandas`, and Jupyter are available in your environment.


In [ ]:
from pathlib import Path
import json
import time

import httpx
import pandas as pd
from IPython.display import Markdown, clear_output, display

BASE_URL = "http://127.0.0.1:8000"
TENANT_ID = "default"
CSV_PATH = Path("../path/to/your_inventory.csv")
POLL_INTERVAL_SECONDS = 1.0
DOWNLOAD_DIR = Path("../notebook_downloads")
PREVIEW_MAX_ROWS = 25
PREVIEW_MAX_GROUPS = 12
PREVIEW_MAX_DUPLICATES = 12

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Update CSV_PATH before running the notebook. Missing file: {CSV_PATH}"
    )

client = httpx.Client(base_url=BASE_URL, timeout=60.0)
display(
    Markdown(
        f"**Base URL:** `{BASE_URL}`  \n"
        f"**Tenant:** `{TENANT_ID}`  \n"
        f"**CSV:** `{CSV_PATH}`  \n"
        f"**Download dir:** `{DOWNLOAD_DIR.resolve()}`"
    )
)


In [ ]:
health_response = client.get("/health")
health_response.raise_for_status()
display(Markdown(f"API health: `{health_response.json()['status']}`"))

source_df = pd.read_csv(CSV_PATH, dtype=str).fillna("")
display(Markdown("## Source file preview"))
display(source_df.head(10))


In [ ]:
with CSV_PATH.open("rb") as csv_file:
    upload_response = client.post(
        "/validate",
        params={"tenant_id": TENANT_ID},
        files={"file": (CSV_PATH.name, csv_file, "text/csv")},
    )

upload_response.raise_for_status()
job_payload = upload_response.json()
JOB_ID = job_payload["job_id"]
display(Markdown(f"## Job created: `{JOB_ID}`"))
job_payload


In [ ]:
STATUS_LABELS = {
    "queued": "Recebido",
    "running": "Em processamento",
    "completed": "Consolidado",
    "failed": "Falha",
}


def _safe_frame(rows: list[dict], empty_message: str) -> pd.DataFrame:
    if rows:
        return pd.DataFrame(rows)
    return pd.DataFrame([{"status": empty_message}])


def _line_number(row_index: int | None) -> str:
    if row_index is None:
        return "-"
    return str(row_index + 2)


def build_duplicates_df(duplicates: list[dict], limit: int = PREVIEW_MAX_DUPLICATES) -> pd.DataFrame:
    rows = []
    for duplicate in duplicates[:limit]:
        rows.append(
            {
                "item": duplicate.get("item"),
                "descricao": duplicate.get("descricao"),
                "count": duplicate.get("count"),
                "row_indices": ", ".join(
                    _line_number(row_index) for row_index in duplicate.get("row_indices", [])
                ),
            }
        )
    return _safe_frame(rows, "No duplicates available for this view")


def build_grouped_problems_df(grouped_problems: dict, limit_groups: int = PREVIEW_MAX_GROUPS) -> pd.DataFrame:
    rows = []
    for code, occurrences in list(grouped_problems.items())[:limit_groups]:
        for occurrence in occurrences:
            rows.append(
                {
                    "code": code,
                    "row_index": occurrence.get("row_index"),
                    "line_number": _line_number(occurrence.get("row_index")),
                    "item": occurrence.get("item"),
                    "descricao": occurrence.get("descricao"),
                    "severity": occurrence.get("severity"),
                    "field": occurrence.get("field"),
                    "message": occurrence.get("message"),
                }
            )
    return _safe_frame(rows, "No grouped problems available for this view")


def build_row_preview_df(row_results: list[dict], limit_rows: int = PREVIEW_MAX_ROWS) -> pd.DataFrame:
    rows = []
    for row in row_results[:limit_rows]:
        issues = row.get("issues", [])
        rows.append(
            {
                "row_index": row.get("row_index"),
                "line_number": _line_number(row.get("row_index")),
                "item": row.get("item"),
                "descricao": row.get("descricao"),
                "issue_count": len(issues),
                "has_errors": row.get("has_errors", False),
                "has_warnings": row.get("has_warnings", False),
            }
        )
    return _safe_frame(rows, "No row preview available for this view")


def render_job_dashboard(payload: dict) -> None:
    status = payload["status"]
    has_preview = bool(payload.get("is_partial_result_available"))
    status_label = STATUS_LABELS.get(status, status)
    total_rows = payload.get("total_rows", 0)
    processed_rows = payload.get("processed_rows", 0)

    clear_output(wait=True)
    display(Markdown(f"## Job monitor: `{payload['job_id']}`"))
    display(
        Markdown(
            f"**Status:** `{status_label}`  \n"
            f"**Step:** `{payload.get('current_step')}`  \n"
            f"**Title:** {payload.get('status_title', '-')}  \n"
            f"**Detail:** {payload.get('status_detail', '-')}"
        )
    )

    progress_df = pd.DataFrame(
        [
            {
                "status": status_label,
                "current_step": payload.get("current_step"),
                "processed_rows": processed_rows,
                "total_rows": total_rows,
                "batch_size": payload.get("batch_size", 0),
                "preview_available": has_preview,
                "file_name": payload.get("file_name"),
                "updated_at": payload.get("updated_at"),
            }
        ]
    )
    display(Markdown("### Execution state"))
    display(progress_df)

    if has_preview and status != "completed":
        partial_summary = payload.get("partial_summary") or {}
        display(Markdown("### Partial preview in update"))
        if partial_summary:
            display(pd.DataFrame([partial_summary]))
        else:
            display(pd.DataFrame([{"status": "Preview flag is on, but the summary is still empty"}]))

        display(Markdown("#### Duplicate items already safe to show"))
        display(build_duplicates_df(payload.get("partial_duplicates", [])))

        display(Markdown("#### Grouped problems from the validated portion"))
        display(build_grouped_problems_df(payload.get("partial_grouped_problems", {})))

        display(Markdown("#### Row results preview"))
        display(build_row_preview_df(payload.get("row_results_preview", [])))
    elif status in {"queued", "running"}:
        display(
            Markdown(
                "> Preview not available yet. The backend is still in the pre-preview phase or has not finished a safe batch yet."
            )
        )

    if status == "completed":
        display(
            Markdown(
                "> Final processing is complete. Run the next cell to fetch the consolidated JSON result from the API."
            )
        )

    if status == "failed":
        display(Markdown(f"**Failure:** `{payload.get('error_message')}`"))


def poll_job_with_preview(job_id: str, poll_interval: float = POLL_INTERVAL_SECONDS) -> dict:
    while True:
        status_response = client.get(f"/jobs/{job_id}")
        status_response.raise_for_status()
        payload = status_response.json()
        render_job_dashboard(payload)
        if payload["status"] in {"completed", "failed"}:
            return payload
        time.sleep(poll_interval)


job_status = poll_job_with_preview(JOB_ID)
job_status


In [ ]:
if job_status["status"] != "completed":
    raise RuntimeError(f"Validation failed: {job_status.get('error_message')}")

result_response = client.get(f"/jobs/{JOB_ID}/result")
result_response.raise_for_status()
report_data = result_response.json()

display(Markdown("## Final consolidated result"))

summary_df = pd.DataFrame([report_data["summary"]])
display(Markdown("### Summary"))
display(summary_df)

row_results_df = build_row_preview_df(report_data["row_results"], limit_rows=PREVIEW_MAX_ROWS)
display(Markdown(f"### Row results preview (first {PREVIEW_MAX_ROWS})"))
display(row_results_df)

duplicates_df = build_duplicates_df(report_data["duplicates"], limit=PREVIEW_MAX_DUPLICATES)
display(Markdown(f"### Duplicates (first {PREVIEW_MAX_DUPLICATES})"))
display(duplicates_df)

grouped_df = build_grouped_problems_df(report_data["grouped_problems"], limit_groups=PREVIEW_MAX_GROUPS)
display(Markdown(f"### Grouped problems (first {PREVIEW_MAX_GROUPS} groups)"))
display(grouped_df)

display(
    Markdown(
        "The full structured result remains available in `report_data` and will also be saved to disk in the next cell."
    )
)


In [ ]:
json_output_path = DOWNLOAD_DIR / f"{JOB_ID}_result.json"
json_output_path.write_text(
    json.dumps(report_data, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

pdf_response = client.get(f"/jobs/{JOB_ID}/report")
pdf_response.raise_for_status()
pdf_output_path = DOWNLOAD_DIR / f"{JOB_ID}_report.pdf"
pdf_output_path.write_bytes(pdf_response.content)

display(Markdown("## Downloaded artifacts"))
display(
    {
        "json_result": str(json_output_path.resolve()),
        "pdf_report": str(pdf_output_path.resolve()),
    }
)


In [ ]:
client.close()
